# GenAI-Kickstart
## Lesson: Prompt Engineering

This lesson is intended to improve your prompt engineering skills.

During this lesson you will learn how to ...
* use a system prompt to define model behaviour
* extend system prompt to create workflows
* specify output format



### Set up the environment 

Before we can start, we have to setup the environment and several default values for model name, model parameter and prompts.  

In [ ]:
import sys
sys.path.append('../../../')

from dotenv import load_dotenv, find_dotenv
from lib.completion import Chat
from lib.validation import validate_environment_for_model
from lib.constants import PROCESSED_DATA_PATH
from pydantic import BaseModel
from lib.completion import generate_completion
import os
import time
from pathlib import Path

load_dotenv(find_dotenv())

print("Environment loaded.")

Environment loaded.


In [2]:
# Path to the book database
BOOK_DB = PROCESSED_DATA_PATH / "books.db"

In [ ]:
# Load the default provider and model. If you want, you can change this.
PROVIDER = os.getenv("DEFAULT_GENERATION_PROVIDER")
GENERATION_MODEL = os.getenv("DEFAULT_GENERATION_MODEL") 
MODEL_ID = f"{PROVIDER}/{GENERATION_MODEL}"

validate_environment_for_model(model=MODEL_ID)

print(f"Using model: {MODEL_ID}")

Using model: gemini/gemini-2.5-flash


### Definition of convenient functions

`Chat` (from `lib.completion`) handles multi-turn conversations — it keeps the message history automatically. `create_bookstore_bot()` is a factory that creates a `Chat` with the book database injected into the system prompt.

In [4]:
# Set default values
DEFAULT_TEMPERATURE = 0.9
DEFAULT_MAX_TOKENS = 500
DEFAULT_SYSTEM_PROMPT = "You are a friendly assistant"

# Load book database content to provide as context to the model
book_db_content = Path(BOOK_DB).read_text(encoding="utf-8")

def create_bookstore_bot(system_prompt: str = DEFAULT_SYSTEM_PROMPT) -> Chat:
    """Creates a Chat instance with the book database injected into the system prompt."""
    full_system_prompt = f"{system_prompt}\n\nBook database:\n{book_db_content}"
    return Chat(
        system_prompt=full_system_prompt,
        model_name=MODEL_ID,
        temperature=DEFAULT_TEMPERATURE,
        max_tokens=DEFAULT_MAX_TOKENS,
    )

In [5]:
def run_conversation(system_prompt: str, customer_prompts: list[str]):
    """Runs a multi-turn conversation with the bookstore bot.

    Parameters
    ----------
    system_prompt : str
        The system prompt for the bot.
    customer_prompts : list[str]
        The list of customer messages to send in order.
    """
    bot = create_bookstore_bot(system_prompt=system_prompt)
    for prompt in customer_prompts:
        print(f'Customer:\n{prompt}\n')
        answer = bot.send(prompt)
        print(f'Bookstore bot:\n{answer}\n')

In [6]:
def order_book(isbn: str):
    """ Orders a book by its ISBN number. 
    
    Parameters
    ----------
    isbn : str
        The ISBN number of the book to order.
    
    Returns 
    -------
    order_status : str
        The status of the order. 
    """
    # Simulate ordering the book
    print(f"Ordering book with ISBN: {isbn}")
    
    # Print "." every 0.5 seconds
    for _ in range(5):
        print(".", end="", flush=True)
        time.sleep(0.5)
    print()
    if isbn == "978-3-51593-12345-6":
        print("Success: You ordered 'A Study in Scarlet'!")
        print("You completed this exercise successfully!")
    else:
        print("Error: Unknown ISBN number.")

### Important notice: History
In this (and only this) exercise, we implemented a history feature so that the conversation can keep on going. Each exercise creates a fresh `Chat` instance, which starts with an empty history. If you mess up mid-exercise, just re-run the cell that creates the bot.

### Exercise 01: Flipped interaction
Your task is to create a simple bookstore chatbot, which is able to gather more information about a book from a customer, if the customer did not provide enough information.

Imagine the following situation:
A customer of a bookstore wants to buy a book, but provides almost no information.  
The customer might ask an employee: *"I'm looking for this one book about a detective."*  
The bookstore employee needs more information in order to help the customer. The employee might ask: "I'm sure I can help you, but I need more information. Do you know the name of the detective or do you know more about the content of the book?"
  
The employee's reaction described here, should now be done by a chatbot. Your task is to provide a system prompt for this bot. 
The bot should:  
* Ask the customer for more information if the provided information is not enough for finding the book.


In [7]:
# Do not change the user prompt.
user_prompt = "I'm looking for this one book about a detective."

# TODO: Define the system prompt of the bookstore bot using the "Flipped Interaction" pattern
system_prompt = """
    You are a helpful bookstore assistant.
    When a customer asks for a book but provides incomplete information, politely ask targeted follow-up questions to clarify what book they mean.
    Request additional details such as the title, author, genre, plot elements, characters, publication period, or anything useful to identify the book.
    Once enough information is provided, suggest possible matches or offer further assistance.
"""

run_conversation(system_prompt, [user_prompt])

Customer:
I'm looking for this one book about a detective.



Bookstore bot:
Ah, a book about a detective! That's a popular genre. To help me narrow it down, could you tell me a little more about it?

For example:
*   Do you remember the **detective's name** or the **author**?
*   What kind of **mysteries** did they solve?
*   Was it set in a particular **time period** or **location**?
*   Any specific **plot elements** or other **characters** you recall?

In our collection, we have:
*   **"A Study in Scarlet" by Arthur Conan Doyle**, which is the first novel featuring the iconic detective Sherlock Holmes. It's set in Victorian London and introduces Dr. Watson.

Does that sound like the one you're looking for, or do you have any more details to share?



### Exercise 02: Basic workflow
Your task is to create a simple bookstore chatbot that can identify a book from a customer description and order it on request.

Imagine the following situation: a customer wants to buy a book, but does not remember its title.  
The customer might ask an employee: *"I'm looking for this book, where Sherlock Holmes and Watson meet the first time."*  
The employee uses their knowledge of books to reply: *"That's 'A Study in Scarlet'! Should I place an order for this book?"*

If the customer confirms (*"Yes, please!"*), the employee proceeds to place the order. The ordering itself is handled by an external API that expects a payload of the form `{"isbn": "978-..."}`, so the bot needs to finish the order flow by replying with a line like `isbn = 978-...`.

Your task is to provide a system prompt for this bot. The bot should:
* Respond to the customer by naming the book title, year of publication and a short summary of the book.
* Ask whether the book should be ordered.
* On confirmation, reply with the template `isbn = *isbn-of-the-book-to-order*`.

**Hints**: The creators of the bot (aka teacher of this workshop) already provided knowledge about books to the bot. You do not need to worry about this.

In [8]:
# TODO: Define the respective system prompt.
system_prompt = "You are a friendly assistant working at a bookstore." \
    "Your task is to help customers to order the book they are looking for." \
    "If a customer does not know the title of the book, use your extensive knowledge to find information" \
    "Provide the title of the book, the year of publication and a short summary of the book" \
    "In addition, ask the customer if you should order the book for them." \
    "If the customers wants to order a book, you reply with the following template: 'isbn = *isbn-of-the-book-to-order*'"


In [9]:
run_conversation(system_prompt, [
    "I'm looking for this book, where Sherlock Holmes and Watson meet the first time.",
    "Yes, I like to order the book.",
])

Customer:
I'm looking for this book, where Sherlock Holmes and Watson meet the first time.

Bookstore bot:
Ah, you're looking for the very beginning of a legendary partnership!

The book you're looking for is:
**Title:** A Study in Scarlet
**Year of Publication:** 1995
**Summary:** This is the first novel that introduces the iconic detective Sherlock Holmes and his companion Dr. John Watson. Set in Victorian London, the story begins with Watson returning from military service in Afghanistan and moving into 221B Baker Street with Holmes, whose brilliant deductive skills soon become evident. Holmes is called upon to investigate a mysterious murder, and through his keen observation and logical reasoning, he unravels a complex tale of love, betrayal, and revenge. Watson, captivated by Holmes's brilliance, decides to chronicle the case, marking the beginning of their legendary partnership in crime-solving.

Would you like me to order this book for you?

Customer:
Yes, I like to order the bo

In [10]:
# TODO: Let's test, if the bot provided the correct isbn for the Sherlock Holmes book.
# Copy the isbn of the last bot response and assign the isbn-variable below to the copied value.
# If you done it correctly, you should see a success message.
isbn = "978-3-51593-12345-6"
order_book(isbn=isbn)

Ordering book with ISBN: 978-3-51593-12345-6
.....
Success: You ordered 'A Study in Scarlet'!
You completed this exercise successfully!


### Exercise 03: Give choices
After the initial customer request, the bot should give the user another choice. In addition to ordering the book, the bot can provide an url to the ebook version of the book as well. Update the system prompt in order to achieve this.

In [11]:
# TODO Update the system prompt to include the option to provide an url to the ebook instead of ordering the book.
system_prompt = """
You are a friendly and knowledgeable assistant working at a bookstore. Your primary task is to help customers find and order the books they are looking for, even if they are unsure of the exact title.
Instructions & Context:
1. Finding the Book:
- If the customer provides the title, locate the book and confirm the details.
- If the customer is unsure of the title, use your extensive knowledge to help them identify it based on any information they provide (e.g., author, plot, genre).
2. Provide Essential Information:
- For each book you identify, provide the following details:
  - Title of the Book
  - Year of Publication
  - Short Summary of the Book
3. Offer Two Clear Choices:
- Choice 1: You order the book for the customer.
- Choice 2: You provide a link to the ebook.
- Formulate these choices in a clear and engaging way, such as: Would you like me to order the physical copy for you, or would you prefer a link to the ebook version?
4. Handling the Customer's Choice:
- If the customer wants the ebook link, provide the URL directly. Use your knowledge to find the ebook link.
- If the customer wants to order the book, respond with the following template: isbn = isbn-of-the-book-to-order

Examples:
Customer: I’m looking for a book about a young wizard who goes to a magical school.
Assistant: I believe you are referring to Harry Potter and the Sorcerer's Stone, published in 1997. It's the first book in the Harry Potter series, where a young boy discovers he's a wizard and attends Hogwarts School of Witchcraft and Wizardry.
Would you like me to order the physical copy for you, or would you prefer a link to the ebook version?

Customer Response: I’d like the ebook, please.
Assistant: Here is the link to the ebook: [ebook-link]

Customer: I want to order The Catcher in the Rye.
Assistant: The Catcher in the Rye, published in 1951, is a classic novel that follows the journey of Holden Caulfield as he navigates teenage angst and alienation in New York City.
Would you like me to order the physical copy for you, or would you prefer a link to the ebook version?

Customer Response: I want to order it.
Assistant: isbn = 9780316769488"
"""

In [12]:
run_conversation(system_prompt, [
    "I'm looking for this book, where Sherlock Holmes and Watson meet the first time.",
    "I love ebooks. Why should I order a book, if I get an ebook for free. Please provide the link!",
])

Customer:
I'm looking for this book, where Sherlock Holmes and Watson meet the first time.

Bookstore bot:
I believe you are referring to **A Study in Scarlet**, published in 1995. This is the first novel that introduces the iconic detective Sherlock Holmes and his companion Dr. John Watson. Set in Victorian London, the story begins with Watson returning from military service in Afghanistan and moving into 221B Baker Street with Holmes, whose brilliant deductive skills soon become evident as they investigate a mysterious murder.

Would you like me to order the physical copy for you, or would you prefer a link to the ebook version?

Customer:
I love ebooks. Why should I order a book, if I get an ebook for free. Please provide the link!

Bookstore bot:
Here is the link to the ebook: https://www.gutenberg.org/ebooks/244



In [13]:
# Test if the old process of ordering a book still works
run_conversation(system_prompt, [
    "I'm looking for this book, where Sherlock Holmes and Watson meet the first time.",
    "Please order the book. I love spending money for stuff I can get for free!",
])

Customer:
I'm looking for this book, where Sherlock Holmes and Watson meet the first time.

Bookstore bot:
I believe you are referring to **A Study in Scarlet**, published in 1995. This is the first novel that introduces the iconic detective Sherlock Holmes and his companion Dr. John Watson. Set in Victorian London, the story begins with Watson returning from military service in Afghanistan and moving into 221B Baker Street with Holmes, whose brilliant deductive skills soon become evident.

Would you like me to order the physical copy for you, or would you prefer a link to the ebook version?

Customer:
Please order the book. I love spending money for stuff I can get for free!

Bookstore bot:
isbn = 978-3-51593-12345-6



### Exercise 04: Define the response format

In all previous exercises the bot replied with free-form text. That's fine for a human reader, but hard to integrate into a real application — any parsing code you write will break as soon as the phrasing changes.

Most modern model providers solve this via a `response_format` parameter: you pass a **schema** describing the expected shape of the reply, and the model is constrained to produce JSON that matches it. A convenient way to define such a schema in Python is a **Pydantic model** — the same class serves as the schema for the model *and* as a validator for the reply.

In this small exercise we'll take a simple customer request ("Can you give me the details of ...") and have the model return a structured object with `title`, `author` and `isbn`.

Your task:
1. Define a Pydantic model `BookInfo` with those three fields.
2. Pass it as `response_format=` to `generate_completion`.
3. Parse the JSON reply back into a `BookInfo` instance.

In [ ]:
# TODO: Define a Pydantic model describing the expected response shape.
class BookInfo(BaseModel):
    title: str
    author: str
    isbn: str


system_prompt = f"""
You are a bookstore assistant. When a customer asks about a book, look it up
in the book database below and return its title, author and ISBN.

Book database:
{book_db_content}
"""

In [15]:
customer_prompt = (
    "Can you give me the details of the book where Sherlock Holmes and Watson "
    "meet for the first time?"
)

raw_reply = generate_completion(
    user_prompt=customer_prompt,
    system_prompt=system_prompt,
    model_name=MODEL_ID,
    temperature=0.0,
    response_format=BookInfo,
)

print(f"Raw model reply:\n{raw_reply}\n")

# TODO: Parse the JSON reply back into a Pydantic instance for reliable access.
# Use the static class method `model_validate_json()`of the BookInfo class in order to validate the raw response!
book = BookInfo.model_validate_json(raw_reply)
print(f"Title:  {book.title}")
print(f"Author: {book.author}")
print(f"ISBN:   {book.isbn}")

Raw model reply:
{"title": "A Study in Scarlet", "author": "Arthur Conan Doyle", "isbn": "978-3-51593-12345-6"}

Title:  A Study in Scarlet
Author: Arthur Conan Doyle
ISBN:   978-3-51593-12345-6
